# Phase 1 — Data EDA

Sanity plots for the primary airport (`KMAN`) and the secondary airport (`KBOI`) to validate that:

1. METAR Parquet files load with the expected schema
2. Wind roses look physically plausible per airport
3. The diurnal cycle is visible (valley thermal winds should show up strongly at KMAN)
4. Raw HRRR 10m wind has a non-trivial bias vs METAR obs — this is the bias our residual model is supposed to learn

This notebook is EDA-only. No feature engineering, no production logic.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, str(Path.cwd().parent / 'src'))
from wind_forecast.config import Airport

AIRPORTS = ['KMAN', 'KBOI']
DATA_ROOT = Path.cwd().parent / 'data'
CONFIG_DIR = Path.cwd().parent / 'config' / 'airports'

## Load METAR

In [ ]:
def load_metar(icao: str) -> pd.DataFrame:
    airport = Airport.load(icao, CONFIG_DIR)
    path = airport.raw_metar_dir(DATA_ROOT) / f'{icao}.parquet'
    df = pd.read_parquet(path)
    df['hour_utc'] = df['valid_utc'].dt.hour
    df['month'] = df['valid_utc'].dt.month
    return df

metar = {icao: load_metar(icao) for icao in AIRPORTS}
for icao, df in metar.items():
    print(f'{icao}: {len(df):,} rows,  {df["valid_utc"].min()} -> {df["valid_utc"].max()}')

## Wind rose

In [ ]:
def wind_rose(ax, df, title):
    mask = df['drct'].notna() & df['sknt'].notna() & (df['sknt'] > 0)
    theta = np.deg2rad(df.loc[mask, 'drct'].to_numpy())
    r = df.loc[mask, 'sknt'].to_numpy()
    ax.hist2d(theta, r, bins=(36, 12), cmap='viridis')
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.set_title(title)

fig, axes = plt.subplots(1, len(AIRPORTS), figsize=(12, 5), subplot_kw={'projection': 'polar'})
for ax, icao in zip(np.atleast_1d(axes), AIRPORTS):
    wind_rose(ax, metar[icao], f'{icao} wind rose')
plt.tight_layout()

## Diurnal cycle

Expect stronger daytime winds at KMAN (valley thermal mixing) than overnight.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for icao, df in metar.items():
    diurnal = df.groupby('hour_utc')['sknt'].mean()
    ax.plot(diurnal.index, diurnal.values, label=icao)
ax.set_xlabel('hour UTC')
ax.set_ylabel('mean wind speed (kt)')
ax.legend()
ax.set_title('Diurnal wind cycle')

## HRRR raw bias (stub)

Once HRRR data is ingested, pair each (cycle, lead) with the matching METAR observation and plot mean (HRRR - obs) by forecast hour. Large hour-of-day bias here is exactly what our residual LightGBM is supposed to absorb.

In [ ]:
# TODO (phase 2): join per-airport HRRR Parquets to METAR on valid_utc
# and compute speed_bias = sqrt(hrrr_u^2 + hrrr_v^2) - sknt
# broken down by (forecast hour, hour of day).